In [38]:
# ===========================================================================
# CELDA 1: CONFIGURACIÓN INICIAL Y RUTAS
# ===========================================================================
import sys
import os

# Definimos la ruta REAL donde está guardado el archivo Fotoapp.py
ruta_del_archivo = r"C:\Users\Caar\Pictures\Fotoapp"

# Agregamos esa ruta al sistema para que Python lo encuentre
sys.path.insert(0, ruta_del_archivo)

import matplotlib
matplotlib.use("Agg") # Backend para que no falle si no hay pantalla
import matplotlib.pyplot as plt
import io
from PIL import Image

# Importamos el módulo (con F mayúscula, como está en la carpeta)
import Fotoapp
from Fotoapp import (
    ENLACES, DIMENSIONES_REDES, _safe_filename,
    ProcessResult, ImageProcessor,
    ImageLoader, ImageVisualizer,
    ContrastProcessor, FilterProcessor, EdgeDetectionProcessor,
    BocetoProcessor, FotoApp,
    redimensionar_para_red, boceto_linea, comparar_bocetos,
)

Fotoapp.ImageVisualizer.ensure_agg_backend()

# Nos movemos a la carpeta del proyecto (PythonProject5) para que las
# imágenes generadas se guarden ahí y no en la carpeta de fotos.
os.chdir(r"C:\Users\Caar\PycharmProjects\PythonProject5")
os.makedirs("imagenes/generadas", exist_ok=True)
os.makedirs("imagenes/prueba", exist_ok=True)

print("Setup OK. El módulo se cargó desde:", sys.modules["Fotoapp"].__file__)

Setup OK. El módulo se cargó desde: C:\Users\Caar\Pictures\Fotoapp\Fotoapp.py


In [39]:
# 1. Aseguramos que existan las carpetas
os.makedirs("imagenes/prueba", exist_ok=True)
os.makedirs("imagenes/generadas", exist_ok=True)

# 2. Generamos la imagen base (fixture) en memoria y la guardamos
buf = io.BytesIO()
Image.new("RGB", (400, 300), (120, 80, 40)).save(buf, format="PNG")
fixture = ImageLoader.load_from_bytes(buf.getvalue())
ImageLoader.save(fixture, "imagenes/prueba/fixture.png")

# 3. Creamos las versiones oscura y clara
oscura = fixture.point(lambda p: int(p * 0.15))
clara  = fixture.point(lambda p: min(255, int(p * 2.8)))

# 4. Procesamos y guardamos
for nombre, foto in [("oscura", oscura), ("clara", clara)]:
    res = ContrastProcessor().process(foto)
    ruta_procesada = f"imagenes/generadas/contraste_{nombre}.png"
    ImageLoader.save(res.procesada, ruta_procesada)

    resultado_grafico = ProcessResult(
        res.original,
        res.procesada,
        res.titulo_izq,
        res.titulo_der,
        f"imagenes/generadas/comparacion_contraste_{nombre}.png"
    )
    ImageVisualizer.compare(resultado_grafico)
    print(f"Guardado con éxito: {ruta_procesada}")

Guardado con éxito: imagenes/generadas/contraste_oscura.png
Guardado con éxito: imagenes/generadas/contraste_clara.png


In [40]:
# ===========================================================================
# CELDA 3: PRUEBA DEL EJERCICIO 1 (REDIMENSIONAR PARA CADA RED)
# ===========================================================================
# Probamos la función con cada una de las redes definidas
for red, esperado in DIMENSIONES_REDES.items():
    # Llamamos a la función pública del ejercicio 1
    img = redimensionar_para_red("imagenes/prueba/fixture.png", red)
    print(f"  Red: {red:10} -> Tamaño obtenido: {img.size} (Esperado: {esperado})")

    # Validamos que el tamaño sea exacto y el modo de color correcto
    assert img.size == esperado, "El tamaño no coincide"
    assert img.mode == "RGB", "El modo de color no es RGB"

    # Guardamos la imagen resultante para tener evidencia
    ImageLoader.save(img, f"imagenes/generadas/red_{red}.png")

# Probamos el manejo de errores con una red inválida
try:
    redimensionar_para_red("imagenes/prueba/fixture.png", "tiktok")
    raise AssertionError("Debería haber lanzado un error")
except ValueError as e:
    print("Manejo de error correcto al pedir red inválida:", e)

# Probamos que no distinga mayúsculas de minúsculas
img_mayus = redimensionar_para_red("imagenes/prueba/fixture.png", "INSTAGRAM")
assert img_mayus.size == (1080, 1080)

print("Ejercicio 1: OK")

  Red: instagram  -> Tamaño obtenido: (1080, 1080) (Esperado: (1080, 1080))
  Red: facebook   -> Tamaño obtenido: (1200, 630) (Esperado: (1200, 630))
  Red: twitter    -> Tamaño obtenido: (1200, 675) (Esperado: (1200, 675))
  Red: youtube    -> Tamaño obtenido: (1280, 720) (Esperado: (1280, 720))
Manejo de error correcto al pedir red inválida: Red no válida: 'tiktok'. Opciones: ['instagram', 'facebook', 'twitter', 'youtube']
Ejercicio 1: OK


In [41]:
# ===========================================================================
# CELDA 4: PRUEBA DEL EJERCICIO 2 (CONTRASTE OSCURA Y CLARA)
# ===========================================================================
# Cargamos la imagen base desde el disco
base = ImageLoader.load_local("imagenes/prueba/fixture.png")

# Generamos dos versiones: una muy oscura (multiplicando por 0.15)
# y una sobreexpuesta (multiplicando por 2.8)
oscura = base.point(lambda p: int(p * 0.15))
clara  = base.point(lambda p: min(255, int(p * 2.8)))

# Procesamos ambas imágenes con el ecualizador de histograma
for nombre, foto in [("oscura", oscura), ("clara", clara)]:
    # Aplicamos el procesador de contraste
    res = ContrastProcessor().process(foto)

    # Guardamos solo la imagen procesada
    ruta_procesada = f"imagenes/generadas/contraste_{nombre}.png"
    ImageLoader.save(res.procesada, ruta_procesada)

    # Preparamos la figura comparativa con el título correspondiente
    resultado_grafico = ProcessResult(
        res.original,
        res.procesada,
        res.titulo_izq,
        res.titulo_der,
        f"imagenes/generadas/comparacion_contraste_{nombre}.png"
    )

    # Generamos y guardamos la comparación lado a lado
    ImageVisualizer.compare(resultado_grafico)
    print(f"Procesada y guardada: {ruta_procesada}")

print("Ejercicio 2: OK")

Procesada y guardada: imagenes/generadas/contraste_oscura.png
Procesada y guardada: imagenes/generadas/contraste_clara.png
Ejercicio 2: OK


In [42]:
# ===========================================================================
# CELDA 5: PRUEBA DEL EJERCICIO 3 (LOS 9 FILTROS POR NÚMERO)
# ===========================================================================
# Instanciamos la aplicación
app = FotoApp(
    contrast_processor=ContrastProcessor(),
    filter_factory=FilterProcessor,
    edge_processor=EdgeDetectionProcessor(),
    boceto_processor=BocetoProcessor(),
)

# Cargamos la imagen de prueba en la aplicación
app.cargar_imagen("imagenes/prueba/fixture.png", red="instagram")

# Creamos el menú numerado del 1 al 9
filtros_num = {str(i + 1): n for i, n in enumerate(FilterProcessor.FILTROS_VALIDOS)}
print("Menú de filtros numerados:")
for num, nombre in filtros_num.items():
    print(f"  {num}: {nombre}")

# Aplicamos cada filtro usando su número correspondiente
for num, nombre in filtros_num.items():
    res = app.aplicar_filtro(nombre)

    # Guardamos la comparación (con un nombre que incluya el filtro)
    resultado_grafico = ProcessResult(
        res.original, res.procesada, res.titulo_izq, res.titulo_der,
        f"imagenes/generadas/f_{_safe_filename(nombre)}.png"
    )
    ImageVisualizer.compare(resultado_grafico)
    print(f"  Filtro {num} ({nombre}) guardado con éxito.")

# Probamos pedir un número que no existe (debe lanzar KeyError)
try:
    filtros_num["99"]
    raise AssertionError("Debería haber lanzado un error")
except KeyError:
    print("Manejo de error correcto al pedir un número de filtro inválido.")

print("Ejercicio 3: OK")

Menú de filtros numerados:
  1: blur
  2: contour
  3: detail
  4: edge enhance
  5: edge enhance more
  6: emboss
  7: find edges
  8: sharpen
  9: smooth
  Filtro 1 (blur) guardado con éxito.
  Filtro 2 (contour) guardado con éxito.
  Filtro 3 (detail) guardado con éxito.
  Filtro 4 (edge enhance) guardado con éxito.
  Filtro 5 (edge enhance more) guardado con éxito.
  Filtro 6 (emboss) guardado con éxito.
  Filtro 7 (find edges) guardado con éxito.
  Filtro 8 (sharpen) guardado con éxito.
  Filtro 9 (smooth) guardado con éxito.
Manejo de error correcto al pedir un número de filtro inválido.
Ejercicio 3: OK


In [43]:
# ===========================================================================
# CELDA 6: PRUEBA DEL EJERCICIO 4 (BOCETO DE LÍNEA CON VARIAS TÉCNICAS)
# ===========================================================================
# Cargamos la imagen base
persona = ImageLoader.load_local("imagenes/prueba/fixture.png")

# 1. Generamos la comparación lado a lado con 3 técnicas
comparar_bocetos(persona, "imagenes/generadas/bocetos_comparacion.png")
print("Comparación de bocetos guardada en: imagenes/generadas/bocetos_comparacion.png")

# 2. Probamos el procesador de boceto
res = app.detectar_boceto()
resultado_grafico = ProcessResult(
    res.original, res.procesada, res.titulo_izq, res.titulo_der,
    "imagenes/generadas/boceto_procesador.png"
)
ImageVisualizer.compare(resultado_grafico)
print("Boceto del procesador guardado en: imagenes/generadas/boceto_procesador.png")

# 3. Guardamos cada técnica individualmente
for t in ("canny", "sobel", "laplaciano", "umbral"):
    b = boceto_linea(persona, t)
    ImageLoader.save(b, f"imagenes/generadas/boceto_{t}.png")
    print(f"Técnica guardada: boceto_{t}.png")

# 4. Probamos el manejo de errores con una técnica inválida
try:
    boceto_linea(persona, "magia")
    raise AssertionError("Debería haber lanzado un error")
except ValueError as e:
    print("Manejo de error correcto:", e)

print("Ejercicio 4: OK")

Comparación de bocetos guardada en: imagenes/generadas/bocetos_comparacion.png
Boceto del procesador guardado en: imagenes/generadas/boceto_procesador.png
Técnica guardada: boceto_canny.png
Técnica guardada: boceto_sobel.png
Técnica guardada: boceto_laplaciano.png
Técnica guardada: boceto_umbral.png
Manejo de error correcto: Técnica no válida: 'magia'. Opciones: canny, sobel, laplaciano, umbral
Ejercicio 4: OK


In [36]:
# ===========================================================================
# CELDA 7: PRUEBA DEL EJERCICIO 5 (VALIDACIÓN DEL MENÚ)
# ===========================================================================
# Creamos una nueva aplicación vacía (sin imagen)
app_vacia = FotoApp(
    contrast_processor=ContrastProcessor(),
    filter_factory=FilterProcessor,
    edge_processor=EdgeDetectionProcessor(),
    boceto_processor=BocetoProcessor(),
)

# Intentamos procesar sin haber cargado imagen. Debe lanzar ValueError.
try:
    app_vacia.aplicar_contraste()
    raise AssertionError("Debería haber lanzado error por falta de imagen")
except ValueError as e:
    print("Validación correcta al intentar procesar sin imagen:", e)

# Ahora cargamos la imagen y vemos que funciona bien
app_vacia.cargar_imagen("imagenes/prueba/fixture.png", red="instagram")
res = app_vacia.aplicar_contraste()
assert res.archivo == "c.png"
print("Ejercicio 5: OK")

Validación correcta al intentar procesar sin imagen: Debe cargar y redimensionar una imagen primero (opción 1).
Ejercicio 5: OK


In [37]:
# ===========================================================================
# CELDA 8: CHECKLIST FINAL (VERIFICACIÓN DE LOS 7 PUNTOS)
# ===========================================================================
# Armamos un diccionario con las condiciones que deben cumplirse
checks = {
    "Punto 1: redimensionar_para_red": callable(redimensionar_para_red),
    "Punto 2: contraste oscura + clara": (
        os.path.isfile("imagenes/generadas/contraste_oscura.png")
        and os.path.isfile("imagenes/generadas/contraste_clara.png")
    ),
    "Punto 3: 9 filtros por número": len(FilterProcessor.FILTROS_VALIDOS) == 9,
    "Punto 4: boceto >=2 técnicas": (
        os.path.isfile("imagenes/generadas/bocetos_comparacion.png")
    ),
    "Punto 5: validación con excepción": True,
    "Punto 6: fotoapp.py + notebook": os.path.isfile(r"C:\Users\Caar\Pictures\Fotoapp\Fotoapp.py"),
    "Punto 7: README + imagenes": (
        os.path.isfile("README.md") and os.path.isdir("imagenes/generadas")
    ),
}

print("=== CHECKLIST DE LOS 7 PUNTOS ===")
for nombre_punto, estado in checks.items():
    # Imprime OK o FALTA según corresponda
    print(("OK    " if estado else "FALTA ") + nombre_punto)

# Verificamos si todo dio verdadero
if all(checks.values()):
    print("\nTODO OK - Listo para subir a GitHub")
else:
    print("\nRevisar los puntos marcados como FALTA")

=== CHECKLIST DE LOS 7 PUNTOS ===
OK    Punto 1: redimensionar_para_red
OK    Punto 2: contraste oscura + clara
OK    Punto 3: 9 filtros por número
OK    Punto 4: boceto >=2 técnicas
OK    Punto 5: validación con excepción
OK    Punto 6: fotoapp.py + notebook
OK    Punto 7: README + imagenes

TODO OK - Listo para subir a GitHub


**CONCLUSIÓN GENERAL DEL PROYECTO FOTOAPP**

**¿Qué es FotoApp?**

FotoApp es un procesador de imágenes para redes sociales construido en Python que demuestra la aplicación práctica de principios SOLID, patrones de diseño y programación orientada a objetos en un caso real.

El programa permite cargar imágenes (desde URLs, rutas locales o claves predefinidas), redimensionarlas a dimensiones exactas de redes sociales (Instagram, Facebook, Twitter, YouTube) y aplicarles diversos procesamientos: contraste, filtros espaciales, detección de bordes y bocetos de línea.

---

**ARQUITECTURA Y PRINCIPIOS APLICADOS**

| Principio | Dónde se aplica |
|---|---|
| SRP (Single Responsibility) | ImageLoader solo carga, ImageVisualizer solo visualiza, FotoApp solo orquesta. |
| OCP (Open/Closed) | Nuevos procesadores se añaden sin modificar FotoApp (ej.: BocetoProcessor). |
| DIP (Dependency Inversion) | FotoApp recibe ImageProcessor abstractos, no clases concretas. |
| ABC + abstractmethod | ImageProcessor obliga a implementar process() en todas las subclases. |
| Dataclass frozen | ProcessResult es inmutable y autodocumentado. |

---

**CÓMO FUNCIONAN LAS FUNCIONES PRINCIPALES**

**1. ImageLoader.load(source) — Carga unificada**

```text
source ("instagram" / URL / ruta local)
    |
    v
ENLACES.get(source.lower(), source) -> resuelve clave a URL
    |
    v
¿startswith("http")?
    |-- SI -> requests.get(url, timeout=15) -> raise_for_status()
    |         -> BytesIO(resp.content)
    |-- NO -> content = ruta local
    |
    v
Image.open(content) -> _normalizar() -> RGB
```

**Clave:** Un solo método maneja 3 formatos de entrada. BytesIO evita escribir en disco. _normalizar() compone transparencias sobre fondo blanco para garantizar siempre RGB.

**2. redimensionar_para_red(ruta, red) — Letterbox exacto (Ejercicio 1)**

```text
red.lower() en DIMENSIONES_REDES? -> NO -> ValueError (fail-fast)
    | SI
    v
ImageLoader.load(ruta) -> imagen RGB
    |
    v
FotoApp._ajustar_a_red(img, red):
    1. copia.thumbnail((w,h), LANCZOS) -> reduce respetando proporcion
    2. lienzo = Image.new("RGB", (w,h), blanco) -> tamano EXACTO
    3. lienzo.paste(copia, centrado) -> letterbox/pillarbox
    |
    v
Devuelve imagen con dimensiones exactas SIN distorsion
```

**Por qué letterbox y no stretch:** stretch deforma la imagen; letterbox añade barras blancas, pero respeta la proporción original.

**3. ContrastProcessor.process(img) — Ecualización de luminancia**

```text
PIL RGB -> np.array -> cv2.cvtColor(RGB2BGR) -> cv2.cvtColor(BGR2YCrCb)
    |
    v
y[:,:,0] = cv2.equalizeHist(y[:,:,0]) -> ecualiza SOLO el canal Y
    |
    v
YCrCb -> BGR -> RGB -> Image.fromarray
    -> ProcessResult(img, res, "Original", "Contraste", "c.png")
```

**Por qué YCrCb:** Ecualizar en RGB alteraría los colores. Al separar luminancia (Y) de crominancia (Cr, Cb), solo se mejora el contraste sin tocar el color.

**4. FilterProcessor(nombre).process(img) — Filtros de Pillow**

```text
__init__(nombre):
    nombre.lower() en FILTROS_VALIDOS? -> NO -> ValueError
    -> self.filter_name = nombre.lower()

process(img):
    filtered = img.filter(FILTROS_VALIDOS[self.filter_name])
    archivo = f"f_{_safe_filename(nombre)}.png"
    -> ProcessResult(img, filtered, "Original", nombre.capitalize(), archivo)
```

**Clave:** Se valida en __init__ (fail-fast) para no fallar después de hacer trabajo. _safe_filename() convierte "edge enhance" en "edge_enhance" para nombres portátiles.

**5. EdgeDetectionProcessor.process(img) — Canny vs Laplaciano**

```text
PIL RGB -> cv2.cvtColor(RGB2GRAY) -> imagen gris (1 canal)
    |
    |-- canny = cv2.Canny(g, 100, 200) -> 1a derivada + histeresis
    |-- laplacian = convertScaleAbs(Laplacian(g)) -> 2a derivada
    |
    v
ProcessResult(canny, laplacian, "Canny", "Laplaciano", "b.png")
```

**Diferencia conceptual:** Canny detecta bordes finos y conectados (más robusto). Laplaciano es isotrópico, pero más sensible al ruido.

**6. boceto_linea(img, tecnica, umbral1, umbral2) — Boceto (Ejercicio 4)**

```text
cv2.cvtColor(RGB2GRAY) -> gris
    |
    |-- "canny"      -> cv2.Canny(g, u1, u2)
    |-- "sobel"      -> magnitud de gradientes X+Y -> threshold
    |-- "laplaciano" -> convertScaleAbs(Laplacian) -> threshold
    |-- "umbral"     -> cv2.threshold(g, OTSU)
    |
    v
b = cv2.bitwise_not(b) -> INVERSION: lineas negras sobre fondo blanco
    |
    v
Image.fromarray(b).convert("RGB")
```

**Por qué invertir:** El estilo "boceto" es líneas oscuras sobre papel blanco. Las técnicas de bordes producen lo contrario (bordes blancos sobre negro).

**7. ImageVisualizer.compare(result) — Comparación y guardado**

```text
plt.subplots(1, 2, figsize=(10,5))
    |
    |-- ax[0].imshow(result.original); set_title(titulo_izq); axis("off")
    |-- ax[1].imshow(result.procesada); set_title(titulo_der); axis("off")
    |
    v
plt.tight_layout() -> savefig(result.archivo, bbox_inches="tight")
    |
    v
plt.show() -> plt.close() -> libera figura (evita acumulacion en RAM)
```

**Orden crítico:** savefig() ANTES de show() porque show() puede consumir la figura. close() libera memoria en bucles.

**8. FotoApp — Controlador con DIP real**

```python
app = FotoApp(
    contrast_processor=ContrastProcessor(),
    filter_factory=FilterProcessor,
    edge_processor=EdgeDetectionProcessor(),
    boceto_processor=BocetoProcessor(),
)
```

**Composition Root:** __main__ es el ÚNICO lugar donde se instancian clases concretas. FotoApp solo conoce la abstracción ImageProcessor, lo que permite cambiar implementaciones sin modificar el controlador (DIP).

**API programática vs menú:** Los métodos cargar_imagen(), aplicar_contraste(), aplicar_filtro(), detectar_bordes(), detectar_boceto() permiten usar la app desde un notebook sin input(). El menú interactivo reutiliza estos métodos (DRY).

---

**FLUJO COMPLETO DE UNA OPERACIÓN TÍPICA**

```text
Usuario elige opcion 1
    -> redimensionar_para_red("instagram", "instagram")
        -> ImageLoader.load("instagram")      [descarga 1200x1200]
        -> _ajustar_a_red(img, "instagram")   [letterbox a 1080x1080]
    -> self.img = resultado

Usuario elige opcion 3 (filtro "blur")
    -> _validar_imagen()                     [lanza ValueError si None]
    -> self._filter_factory("blur")          [FilterProcessor("blur")]
    -> processor.process(self.img)           [aplica ImageFilter.BLUR]
    -> ProcessResult(img, filtered, "Original", "Blur", "f_blur.png")
    -> ImageVisualizer.compare(result)       [dibuja, guarda PNG, muestra,
                                               cierra]
```

---

**RESUMEN DE DECISIONES CLAVE**

| Decisión | Razón |
|---|---|
| BytesIO en vez de archivos temporales | Velocidad, sin permisos, todo en RAM. |
| @dataclass(frozen=True) | Inmutabilidad + hashabilidad + boilerplate automático. |
| ABC + @abstractmethod | Falla en runtime si una subclase olvida process(). |
| Callable[[str], ImageProcessor] para el factory | FilterProcessor requiere filter_name en __init__. |
| _normalizar() compartido | DRY: las 3 rutas de carga devuelven siempre RGB. |
| ensure_agg_backend() | Permite savefig() en Colab/servidores sin pantalla. |
| plt.close() tras cada comparación | Evita acumulación de figuras en RAM. |
| __all__ explícito | Controla qué se exporta con "from fotoapp import *". |

---

**CONCLUSIÓN FINAL**

FotoApp demuestra que una arquitectura bien diseñada no es burocracia, sino inversión: cada clase tiene una responsabilidad clara, cada principio SOLID resuelve un problema concreto (extensibilidad, testabilidad, bajo acoplamiento), y cada función pública cumple un contrato tipado que documenta su propósito.

El resultado es un sistema donde añadir un nuevo procesador (como BocetoProcessor) no requiere tocar el controlador, donde los errores fallan temprano con mensajes claros, y donde el código es reutilizable tanto desde un menú interactivo como desde un notebook de pruebas.